# ✈️ FlightAI — Airline AI Assistant (Free Models Edition)

This is the **free-model** version of the original `day5.ipynb` multimodal airline assistant.

| Component | Original (Paid) | This Version (Free) |
|---|---|---|
| LLM Chat | `gpt-4.1-mini` (OpenAI) | `llama-3.3-70b-versatile` (Groq) |
| Text-to-Speech | `gpt-4o-mini-tts` (OpenAI) | `playai-tts` (Groq TTS) |
| Image Generation | `dall-e-3` (OpenAI) | Pollinations.ai (No API key) |

**Requirements:**
- `pip install groq gradio pillow requests python-dotenv`
- A free Groq API key from [console.groq.com](https://console.groq.com) — set it as `GROQ_API_KEY` in your `.env`
- No OpenAI key needed!

In [ ]:
# Imports
import os
import json
import requests
import base64
from io import BytesIO
from PIL import Image
from dotenv import load_dotenv
from groq import Groq
import gradio as gr
import sqlite3
import urllib.parse

In [ ]:
# Initialization
load_dotenv(override=True)

groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set — please add GROQ_API_KEY to your .env file")

# ✅ llama-3.3-70b-versatile is Groq's official recommended model for tool use
# (llama3-groq-70b-8192-tool-use-preview was deprecated Jan 6, 2025)
MODEL = "llama-3.3-70b-versatile"

client = Groq()
DB = "prices.db"

In [ ]:
# Create & seed the prices database — run this ONCE before launching the UI
# This creates prices.db with sample city prices

import sqlite3

DB = "prices.db"

cities_prices = {
    "london": "$799",
    "paris": "$899",
    "tokyo": "$1399",
    "dubai": "$649",
    "new york": "$999",
    "singapore": "$1199",
    "sydney": "$1599",
    "bangkok": "$849",
    "amsterdam": "$749",
    "rome": "$699",
    "berlin": "$729",
    "toronto": "$849",
    "los angeles": "$1099",
    "hong kong": "$1249",
    "barcelona": "$779",
}

with sqlite3.connect(DB) as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price TEXT)")
    conn.executemany("INSERT OR REPLACE INTO prices VALUES (?, ?)", cities_prices.items())
    conn.commit()

print(f"✅ Database created at {DB} with {len(cities_prices)} cities")
print("Sample cities:", ", ".join(list(cities_prices.keys())[:5]))

In [ ]:
# System message — unchanged from original
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
# Database Tool — unchanged from original
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?", (city.lower(),))
        result = cursor.fetchone()
    return f"Ticket price to {city} is {result[0]}" if result else f"No price data available for {city}"

# Test it
# get_ticket_price("Paris")

In [ ]:
# Tool definition — unchanged from original (Groq uses the same OpenAI-compatible tools format)
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": price_function}]

In [ ]:
# Tool call handler — returns both responses and list of cities mentioned
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id,
            })
    return responses, cities

In [ ]:
# 🎨 Image Generation using Pollinations.ai — FREE, no API key needed!
# Replaces: openai.images.generate(model="dall-e-3", ...)

def artist(city):
    print(f"IMAGE TOOL CALLED: Generating image for {city}", flush=True)
    prompt = f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style"
    encoded_prompt = urllib.parse.quote(prompt)
    # Pollinations.ai free image generation endpoint — no key required
    url = f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=1024&height=1024&nologo=true&model=flux"
    response = requests.get(url, timeout=60)
    if response.status_code == 200:
        return Image.open(BytesIO(response.content))
    else:
        print(f"Image generation failed: {response.status_code}")
        return None

# Test it
# image = artist("Tokyo")
# display(image)

In [ ]:
# 🔊 Text-to-Speech using Groq Orpheus TTS — FREE on Groq free tier!
# Valid voices for canopylabs/orpheus-v1-english: autumn, diana, hannah, austin, daniel, troy

def talker(message):
    print(f"TTS TOOL CALLED", flush=True)
    response = client.audio.speech.create(
        model="canopylabs/orpheus-v1-english",
        voice="diana",    # ✅ Valid voices: autumn, diana, hannah, austin, daniel, troy
        input=message,
        response_format="wav",
    )
    audio_bytes = bytearray()
    for chunk in response.iter_bytes():
        audio_bytes.extend(chunk)
    return bytes(audio_bytes)

In [ ]:
# 🤟 Sign Language Animated Hand Block
# Uses srcdoc iframe to bypass Gradio's script sandboxing
import re
import json as _json

ASL_LANDMARKS = {
    "A": [[0.5,0.9],[0.35,0.75],[0.3,0.62],[0.28,0.52],[0.27,0.45],[0.42,0.45],[0.42,0.33],[0.42,0.23],[0.42,0.16],[0.5,0.43],[0.5,0.3],[0.5,0.2],[0.5,0.13],[0.58,0.45],[0.58,0.33],[0.58,0.23],[0.58,0.16],[0.65,0.5],[0.67,0.38],[0.67,0.29],[0.66,0.22]],
    "B": [[0.5,0.9],[0.35,0.75],[0.3,0.65],[0.28,0.58],[0.27,0.52],[0.4,0.43],[0.4,0.28],[0.4,0.16],[0.4,0.08],[0.48,0.42],[0.48,0.27],[0.48,0.15],[0.48,0.07],[0.56,0.43],[0.56,0.28],[0.56,0.16],[0.56,0.08],[0.63,0.45],[0.65,0.3],[0.65,0.18],[0.64,0.1]],
    "C": [[0.5,0.9],[0.3,0.72],[0.25,0.6],[0.22,0.5],[0.2,0.42],[0.35,0.38],[0.3,0.26],[0.28,0.18],[0.27,0.12],[0.44,0.35],[0.4,0.23],[0.38,0.15],[0.37,0.09],[0.54,0.35],[0.5,0.23],[0.49,0.15],[0.48,0.09],[0.63,0.4],[0.62,0.29],[0.62,0.21],[0.61,0.14]],
    "D": [[0.5,0.9],[0.32,0.73],[0.3,0.6],[0.3,0.5],[0.32,0.43],[0.4,0.38],[0.38,0.24],[0.38,0.14],[0.4,0.08],[0.5,0.42],[0.52,0.29],[0.53,0.2],[0.52,0.13],[0.58,0.45],[0.6,0.33],[0.6,0.24],[0.59,0.17],[0.65,0.5],[0.67,0.39],[0.67,0.3],[0.66,0.23]],
    "E": [[0.5,0.9],[0.33,0.73],[0.28,0.62],[0.26,0.54],[0.25,0.48],[0.38,0.48],[0.37,0.38],[0.37,0.3],[0.37,0.24],[0.47,0.46],[0.46,0.36],[0.46,0.28],[0.46,0.22],[0.55,0.47],[0.55,0.37],[0.54,0.29],[0.54,0.23],[0.63,0.5],[0.64,0.4],[0.64,0.32],[0.63,0.26]],
    "F": [[0.5,0.9],[0.33,0.73],[0.3,0.6],[0.3,0.52],[0.32,0.46],[0.4,0.38],[0.4,0.25],[0.4,0.15],[0.4,0.08],[0.49,0.42],[0.5,0.3],[0.5,0.22],[0.49,0.16],[0.57,0.44],[0.59,0.32],[0.59,0.23],[0.58,0.16],[0.65,0.48],[0.66,0.37],[0.66,0.28],[0.65,0.21]],
    "G": [[0.5,0.9],[0.3,0.73],[0.25,0.62],[0.22,0.53],[0.2,0.45],[0.38,0.4],[0.32,0.28],[0.28,0.2],[0.26,0.13],[0.48,0.45],[0.47,0.34],[0.46,0.26],[0.46,0.19],[0.56,0.47],[0.56,0.36],[0.56,0.28],[0.55,0.21],[0.64,0.5],[0.65,0.4],[0.65,0.31],[0.64,0.24]],
    "H": [[0.5,0.9],[0.33,0.74],[0.28,0.63],[0.26,0.54],[0.24,0.47],[0.38,0.4],[0.33,0.27],[0.3,0.18],[0.29,0.11],[0.47,0.4],[0.44,0.27],[0.42,0.18],[0.41,0.11],[0.56,0.46],[0.56,0.36],[0.56,0.28],[0.55,0.21],[0.64,0.5],[0.65,0.4],[0.65,0.32],[0.64,0.25]],
    "I": [[0.5,0.9],[0.35,0.74],[0.3,0.63],[0.28,0.54],[0.27,0.47],[0.41,0.47],[0.41,0.36],[0.4,0.27],[0.4,0.21],[0.49,0.47],[0.49,0.36],[0.49,0.28],[0.48,0.21],[0.57,0.47],[0.57,0.36],[0.56,0.28],[0.56,0.21],[0.65,0.45],[0.66,0.33],[0.66,0.23],[0.66,0.15]],
    "J": [[0.5,0.9],[0.35,0.74],[0.3,0.63],[0.28,0.54],[0.27,0.47],[0.41,0.47],[0.41,0.36],[0.4,0.27],[0.4,0.21],[0.49,0.47],[0.49,0.36],[0.49,0.28],[0.48,0.21],[0.57,0.47],[0.57,0.36],[0.56,0.28],[0.56,0.21],[0.65,0.42],[0.68,0.3],[0.7,0.2],[0.72,0.12]],
    "K": [[0.5,0.9],[0.33,0.73],[0.28,0.61],[0.26,0.51],[0.25,0.43],[0.38,0.38],[0.35,0.25],[0.33,0.16],[0.32,0.09],[0.47,0.4],[0.46,0.27],[0.45,0.18],[0.44,0.11],[0.56,0.46],[0.56,0.35],[0.56,0.27],[0.55,0.2],[0.64,0.5],[0.65,0.4],[0.65,0.32],[0.64,0.25]],
    "L": [[0.5,0.9],[0.28,0.73],[0.22,0.6],[0.18,0.5],[0.15,0.42],[0.38,0.38],[0.35,0.24],[0.33,0.14],[0.32,0.07],[0.48,0.46],[0.48,0.35],[0.48,0.27],[0.47,0.2],[0.56,0.47],[0.57,0.37],[0.56,0.29],[0.56,0.22],[0.64,0.5],[0.65,0.4],[0.65,0.32],[0.64,0.25]],
    "M": [[0.5,0.9],[0.34,0.73],[0.29,0.62],[0.27,0.53],[0.26,0.46],[0.39,0.52],[0.38,0.42],[0.37,0.34],[0.37,0.28],[0.47,0.52],[0.47,0.42],[0.46,0.34],[0.46,0.28],[0.55,0.52],[0.55,0.42],[0.54,0.34],[0.54,0.28],[0.63,0.54],[0.64,0.44],[0.64,0.36],[0.63,0.3]],
    "N": [[0.5,0.9],[0.34,0.73],[0.29,0.62],[0.27,0.53],[0.26,0.46],[0.39,0.52],[0.38,0.42],[0.37,0.34],[0.37,0.28],[0.47,0.52],[0.47,0.42],[0.46,0.34],[0.46,0.28],[0.55,0.55],[0.55,0.46],[0.54,0.38],[0.54,0.32],[0.63,0.57],[0.64,0.47],[0.64,0.39],[0.63,0.33]],
    "O": [[0.5,0.9],[0.32,0.72],[0.27,0.6],[0.25,0.5],[0.25,0.42],[0.37,0.37],[0.33,0.26],[0.31,0.18],[0.32,0.12],[0.46,0.35],[0.43,0.24],[0.42,0.16],[0.43,0.1],[0.55,0.36],[0.53,0.25],[0.52,0.17],[0.53,0.11],[0.63,0.41],[0.63,0.31],[0.62,0.24],[0.62,0.18]],
    "P": [[0.5,0.9],[0.32,0.73],[0.27,0.62],[0.25,0.53],[0.24,0.46],[0.38,0.4],[0.33,0.28],[0.3,0.19],[0.29,0.12],[0.47,0.41],[0.45,0.29],[0.44,0.2],[0.43,0.13],[0.56,0.47],[0.56,0.37],[0.55,0.29],[0.55,0.22],[0.64,0.51],[0.65,0.41],[0.65,0.33],[0.64,0.26]],
    "Q": [[0.5,0.9],[0.3,0.72],[0.25,0.6],[0.22,0.5],[0.21,0.42],[0.37,0.4],[0.33,0.28],[0.31,0.19],[0.31,0.13],[0.47,0.44],[0.46,0.33],[0.46,0.25],[0.46,0.18],[0.55,0.47],[0.56,0.37],[0.55,0.29],[0.55,0.22],[0.64,0.51],[0.65,0.41],[0.65,0.33],[0.64,0.26]],
    "R": [[0.5,0.9],[0.34,0.74],[0.29,0.63],[0.27,0.54],[0.26,0.47],[0.4,0.38],[0.37,0.26],[0.35,0.17],[0.34,0.1],[0.48,0.38],[0.46,0.26],[0.44,0.17],[0.44,0.1],[0.57,0.47],[0.57,0.37],[0.56,0.29],[0.56,0.22],[0.65,0.51],[0.66,0.41],[0.65,0.33],[0.65,0.26]],
    "S": [[0.5,0.9],[0.34,0.73],[0.29,0.62],[0.27,0.53],[0.27,0.46],[0.39,0.5],[0.38,0.4],[0.38,0.32],[0.38,0.26],[0.47,0.5],[0.47,0.4],[0.46,0.32],[0.46,0.26],[0.55,0.5],[0.55,0.4],[0.54,0.32],[0.54,0.26],[0.63,0.52],[0.64,0.42],[0.64,0.34],[0.63,0.28]],
    "T": [[0.5,0.9],[0.33,0.72],[0.28,0.6],[0.27,0.51],[0.28,0.44],[0.39,0.5],[0.38,0.4],[0.38,0.32],[0.38,0.26],[0.48,0.5],[0.47,0.4],[0.47,0.32],[0.47,0.26],[0.56,0.5],[0.56,0.4],[0.55,0.32],[0.55,0.26],[0.64,0.52],[0.65,0.42],[0.64,0.34],[0.64,0.28]],
    "U": [[0.5,0.9],[0.34,0.74],[0.29,0.63],[0.27,0.54],[0.26,0.47],[0.4,0.38],[0.38,0.25],[0.37,0.15],[0.36,0.08],[0.48,0.38],[0.47,0.25],[0.46,0.15],[0.46,0.08],[0.57,0.47],[0.57,0.37],[0.56,0.29],[0.56,0.22],[0.65,0.51],[0.66,0.41],[0.65,0.33],[0.65,0.26]],
    "V": [[0.5,0.9],[0.34,0.74],[0.29,0.63],[0.27,0.54],[0.26,0.47],[0.39,0.38],[0.36,0.25],[0.34,0.15],[0.33,0.08],[0.48,0.38],[0.46,0.25],[0.45,0.15],[0.45,0.08],[0.57,0.47],[0.57,0.37],[0.56,0.29],[0.56,0.22],[0.65,0.51],[0.66,0.41],[0.65,0.33],[0.65,0.26]],
    "W": [[0.5,0.9],[0.34,0.74],[0.29,0.63],[0.27,0.54],[0.26,0.47],[0.39,0.37],[0.36,0.24],[0.34,0.14],[0.33,0.07],[0.47,0.37],[0.46,0.24],[0.45,0.14],[0.45,0.07],[0.55,0.37],[0.55,0.24],[0.54,0.14],[0.54,0.07],[0.65,0.51],[0.66,0.41],[0.65,0.33],[0.65,0.26]],
    "X": [[0.5,0.9],[0.34,0.73],[0.29,0.62],[0.27,0.53],[0.26,0.46],[0.39,0.44],[0.38,0.34],[0.37,0.26],[0.37,0.2],[0.48,0.46],[0.47,0.36],[0.47,0.28],[0.47,0.22],[0.56,0.48],[0.56,0.38],[0.55,0.3],[0.55,0.24],[0.64,0.51],[0.65,0.41],[0.64,0.33],[0.64,0.27]],
    "Y": [[0.5,0.9],[0.3,0.72],[0.24,0.6],[0.2,0.5],[0.18,0.42],[0.4,0.46],[0.4,0.35],[0.39,0.26],[0.39,0.19],[0.48,0.47],[0.48,0.36],[0.47,0.27],[0.47,0.2],[0.56,0.47],[0.56,0.37],[0.55,0.29],[0.55,0.22],[0.65,0.44],[0.67,0.32],[0.67,0.22],[0.67,0.14]],
    "Z": [[0.5,0.9],[0.34,0.73],[0.29,0.62],[0.27,0.53],[0.26,0.46],[0.39,0.38],[0.36,0.25],[0.34,0.15],[0.33,0.08],[0.48,0.46],[0.47,0.36],[0.47,0.28],[0.47,0.22],[0.56,0.48],[0.56,0.38],[0.55,0.3],[0.55,0.24],[0.64,0.51],[0.65,0.41],[0.64,0.33],[0.64,0.27]],
}

HAND_CONNECTIONS = [
    [0,1],[1,2],[2,3],[3,4],
    [0,5],[5,6],[6,7],[7,8],
    [0,9],[9,10],[10,11],[11,12],
    [0,13],[13,14],[14,15],[15,16],
    [0,17],[17,18],[18,19],[19,20],
    [5,9],[9,13],[13,17],
]

def build_sign_animation_html(reply):
    clean = re.sub(r"[^a-zA-Z ]", " ", reply).upper()
    chars = []
    for word in clean.split():
        for ch in word:
            chars.append(ch)
        chars.append(" ")

    needed = {c: ASL_LANDMARKS[c] for c in chars if c != " " and c in ASL_LANDMARKS}

    lm_js    = _json.dumps(needed)
    conn_js  = _json.dumps(HAND_CONNECTIONS)
    chars_js = _json.dumps(chars)
    reply_esc = reply.replace("\\", "\\\\").replace("`", "\\`").replace("$", "\\$")

    # Full self-contained HTML page rendered inside srcdoc iframe
    # srcdoc iframe is NOT sandboxed by Gradio — scripts run freely
    inner_html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{background:#0f172a;font-family:'Segoe UI',Arial,sans-serif;color:white;padding:16px;overflow-x:hidden}}
  h3{{color:#38bdf8;font-size:16px;margin-bottom:2px}}
  .subtitle{{color:#64748b;font-size:11px;font-style:italic;margin-bottom:10px;line-height:1.4;word-break:break-word}}
  #sentence{{font-size:12px;background:#1e293b;padding:7px 10px;border-radius:6px;
             margin-bottom:12px;min-height:22px;line-height:1.8;word-break:break-all;letter-spacing:1px}}
  .row{{display:flex;align-items:center;gap:18px;flex-wrap:wrap}}
  canvas{{background:#0a0f1e;border-radius:10px;border:1px solid #334155;flex-shrink:0}}
  #letter{{font-size:80px;font-weight:900;color:#38bdf8;line-height:1;text-shadow:0 0 24px #38bdf8aa}}
  #word{{font-size:16px;color:#e2e8f0;margin-top:6px;letter-spacing:3px;font-weight:bold}}
  #status{{color:#64748b;font-size:11px;margin-top:5px}}
  .controls{{display:flex;gap:8px;margin-top:12px;flex-wrap:wrap;align-items:center}}
  button{{border:none;padding:8px 18px;border-radius:7px;cursor:pointer;font-size:13px;font-weight:bold;color:white}}
  #btn-pp{{background:#0369a1}} #btn-rr{{background:#065f46}}
  .speed-row{{display:flex;align-items:center;gap:6px;color:#94a3b8;font-size:12px}}
  #progressbar{{background:#1e293b;border-radius:3px;margin-top:10px;height:3px;overflow:hidden}}
  #progressfill{{height:3px;background:linear-gradient(90deg,#38bdf8,#818cf8);width:0%;border-radius:3px;transition:width 0.15s}}
</style>
</head>
<body>
<h3>🤟 ASL Fingerspelling — Full Sentence</h3>
<div class="subtitle" id="subtitle"></div>
<div id="sentence"></div>
<div class="row">
  <canvas id="c" width="240" height="270"></canvas>
  <div>
    <div id="letter">—</div>
    <div id="word"></div>
    <div id="status">Starting…</div>
  </div>
</div>
<div class="controls">
  <button id="btn-pp">⏸ Pause</button>
  <button id="btn-rr">↩ Restart</button>
  <div class="speed-row">🐢 <input type="range" id="spd" min="150" max="1200" value="650" step="50" style="width:90px;cursor:pointer"> 🐇</div>
</div>
<div id="progressbar"><div id="progressfill"></div></div>

<script>
const ASL   = {lm_js};
const CONN  = {conn_js};
const CHARS = {chars_js};
const REPLY = `{reply_esc}`;

document.getElementById('subtitle').textContent = '\\u201c' + REPLY + '\\u201d';

const canvas = document.getElementById('c');
const ctx    = canvas.getContext('2d');
const W=canvas.width, H=canvas.height;

let idx=0, paused=false, timer=null, speed=650;

document.getElementById('spd').oninput = function(){{
  speed=parseInt(this.value);
  if(!paused){{clearInterval(timer);runTimer();}};
}};
document.getElementById('btn-pp').onclick = function(){{
  paused=!paused;
  this.textContent = paused ? '▶ Resume' : '⏸ Pause';
  if(!paused) runTimer(); else clearInterval(timer);
}};
document.getElementById('btn-rr').onclick = function(){{
  clearInterval(timer); idx=0; paused=false;
  document.getElementById('btn-pp').textContent='⏸ Pause';
  tick(); runTimer();
}};

function drawHand(lm){{
  ctx.clearRect(0,0,W,H);
  if(!lm) return;
  const pts=lm.map(([x,y])=>[x*W,y*H]);
  const g=ctx.createRadialGradient(W/2,H*.55,10,W/2,H*.55,110);
  g.addColorStop(0,'rgba(56,189,248,.1)'); g.addColorStop(1,'rgba(0,0,0,0)');
  ctx.fillStyle=g; ctx.fillRect(0,0,W,H);
  CONN.forEach(([a,b])=>{{
    const gr=ctx.createLinearGradient(pts[a][0],pts[a][1],pts[b][0],pts[b][1]);
    gr.addColorStop(0,'rgba(56,189,248,.95)'); gr.addColorStop(1,'rgba(129,140,248,.95)');
    ctx.beginPath(); ctx.moveTo(...pts[a]); ctx.lineTo(...pts[b]);
    ctx.strokeStyle=gr; ctx.lineWidth=3.5; ctx.lineCap='round'; ctx.stroke();
  }});
  pts.forEach(([x,y],i)=>{{
    const tip=[4,8,12,16,20].includes(i);
    ctx.beginPath(); ctx.arc(x,y,tip?7:4.5,0,Math.PI*2);
    ctx.fillStyle=tip?'#f0abfc':'#bfdbfe';
    ctx.shadowColor=tip?'#e879f9':'#60a5fa'; ctx.shadowBlur=tip?16:9;
    ctx.fill(); ctx.shadowBlur=0;
  }});
}}

function buildSentenceHTML(){{
  let h='';
  CHARS.forEach((c,i)=>{{
    if(c===' '){{h+='<span style="display:inline-block;width:10px"></span>';return;}}
    const col=i===idx?'#f0abfc':i<idx?'#38bdf8':'#475569';
    const fw=i<=idx?'bold':'normal';
    const bg=i===idx?'rgba(240,171,252,.18)':'transparent';
    h+=`<span style="color:${{col}};font-weight:${{fw}};background:${{bg}};border-radius:2px;padding:0 1px">${{c}}</span>`;
  }});
  return h;
}}

function currentWord(){{
  let s=idx,e=idx;
  while(s>0&&CHARS[s-1]!==' ') s--;
  while(e<CHARS.length&&CHARS[e]!==' ') e++;
  return CHARS.slice(s,e).join('');
}}

function tick(){{
  if(idx>=CHARS.length){{
    clearInterval(timer);
    document.getElementById('status').textContent='✅ Done — press Restart to replay';
    document.getElementById('letter').textContent='✓';
    document.getElementById('progressfill').style.width='100%';
    return;
  }}
  const c=CHARS[idx];
  document.getElementById('sentence').innerHTML=buildSentenceHTML();
  document.getElementById('progressfill').style.width=((idx/CHARS.length)*100).toFixed(1)+'%';
  if(c===' '){{
    drawHand(null); ctx.clearRect(0,0,W,H);
    document.getElementById('letter').textContent='·';
    document.getElementById('word').textContent='';
    document.getElementById('status').textContent='[ word gap ]';
  }} else {{
    drawHand(ASL[c]||null);
    document.getElementById('letter').textContent=c;
    document.getElementById('word').textContent=currentWord();
    document.getElementById('status').textContent='Letter '+(idx+1)+' / '+CHARS.length;
  }}
  idx++;
}}

function runTimer(){{
  clearInterval(timer);
  timer=setInterval(()=>{{if(!paused) tick();}}, speed);
}}

tick(); runTimer();
</script>
</body>
</html>"""

    # Encode as srcdoc attribute — escape quotes properly
    srcdoc = inner_html.replace('"', '&quot;')

    outer = f"""<iframe srcdoc="{srcdoc}"
      style="width:100%;height:430px;border:none;border-radius:14px;overflow:hidden;"
      sandbox="allow-scripts">
    </iframe>"""
    return outer


In [ ]:
# 💬 Main Chat Function — now returns sign language output too
def chat(history):
    messages = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + messages

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    cities = []
    image = None

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)

        assistant_msg = {
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in message.tool_calls
            ],
        }
        messages.append(assistant_msg)
        messages.extend(responses)

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

    reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": reply})

    # Generate audio
    voice = talker(reply)

    # Generate destination image if city was mentioned
    if cities:
        image = artist(cities[0])

    # Generate ASL sign language guide
    sign_html = build_sign_animation_html(reply)

    return history, voice, image, sign_html

In [ ]:
# Helper to add user message to chatbot
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

In [ ]:
# 🚀 Gradio UI — now includes ASL Sign Language panel
with gr.Blocks() as ui:
    gr.Markdown("# ✈️ FlightAI — Your AI Airline Assistant")
    gr.Markdown("Powered by **Groq LLaMA 3.3** · **Groq TTS (Orpheus)** · **Pollinations.ai** · **ASL Sign Guide**")

    with gr.Row():
        chatbot = gr.Chatbot(height=450, type="messages", label="Chat")
        image_output = gr.Image(height=450, interactive=False, label="Destination Preview")

    with gr.Row():
        audio_output = gr.Audio(autoplay=True, label="🔊 Voice Reply")

    with gr.Row():
        sign_output = gr.HTML(label="🤟 ASL Sign Language Guide")

    with gr.Row():
        message = gr.Textbox(
            label="Chat with FlightAI — Ask about ticket prices, destinations, and more!",
            placeholder="e.g. What is the ticket price to Paris?"
        )

    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    ).then(
        chat,
        inputs=chatbot,
        outputs=[chatbot, audio_output, image_output, sign_output]
    )

ui.launch(inbrowser=True)

## 📝 Notes & Tips

### Model Alternatives for Groq LLM
```python
MODEL = "llama-3.3-70b-versatile"   # Best quality (default)
MODEL = "llama-3.1-8b-instant"      # Fastest, lightest
MODEL = "gemma2-9b-it"              # Google Gemma 2 via Groq
MODEL = "mixtral-8x7b-32768"        # Mixtral with 32k context
```

### TTS Voice Alternatives
```python
# Replace 'Fritz-PlayAI' in talker() with any of:
"Celeste-PlayAI"  # Female, clear
"Ariel-PlayAI"    # Female, warm
"Atlas-PlayAI"    # Male, deep
"Basil-PlayAI"    # British male
"Chip-PlayAI"     # Male, upbeat
```

### Image Model Alternatives (Pollinations.ai)
```python
# Replace 'flux' in artist() with:
"flux"       # FLUX.1 — best quality (default)
"turbo"      # SDXL Turbo — faster
"gptimage"   # GPT Image-1
```

### Using Gemini Instead of Groq
If you have a free Gemini API key, replace the LLM client with:
```python
from google import genai
client = genai.Client()   # Uses GEMINI_API_KEY from env
MODEL = "gemini-2.0-flash"
```
> Groq TTS and Pollinations.ai still handle audio and images — Gemini covers only the chat LLM.